In [ ]:
"""
This function generates time-based features for a forecast target time, given:

idx — the observation timestamps (when you have the data)
h — how far ahead you're forecasting (horizon bucket, 1-based)
It shifts each timestamp forward by h × OUTPUT_RESOLUTION minutes to get the target time, then returns 8 cyclical/categorical features describing that target time:

Feature	Description
sin/cos of hour	Smooth encoding of time-of-day (24h cycle)
sin/cos of day-of-week	Smooth encoding of day-of-week (7-day cycle)
is_weekend	1 if target falls on Sat/Sun
is_evening	1 if target is 17:00–21:00
is_daytime	1 if target is 07:00–17:00
is_overnight	1 if target is 21:00–07:00
The key insight is that it describes when the prediction is for, not when the input was observed — so the model knows the temporal context of what it's trying to predict.
"""



def _compute_target_time_feats(idx: pd.DatetimeIndex, h: int) -> np.ndarray:
    """Return (N, 8) float32 array of target-time features for horizon h.

    h is a horizon bucket index (1-based). Each bucket is OUTPUT_RESOLUTION minutes wide.
    """
    _interval = int(os.environ["OUTPUT_RESOLUTION"])
    total_m = idx.hour * 60 + idx.minute + h * _interval
    t_hr = (total_m % 1440) / 60.0
    t_dow = (idx.dayofweek + total_m // 1440) % 7
    return np.column_stack([
        np.sin(2 * np.pi * t_hr / 24.0).astype(np.float32),
        np.cos(2 * np.pi * t_hr / 24.0).astype(np.float32),
        np.sin(2 * np.pi * t_dow / 7.0).astype(np.float32),
        np.cos(2 * np.pi * t_dow / 7.0).astype(np.float32),
        (t_dow >= 5).astype(np.float32),
        ((t_hr >= 17.0) & (t_hr < 21.0)).astype(np.float32),
        ((t_hr >= 7.0) & (t_hr < 17.0)).astype(np.float32),
        ((t_hr < 7.0) | (t_hr >= 21.0)).astype(np.float32),
    ])



